# Лабораторная работа 2. Нейрон и однослойная нейронная сеть

**Задание:**
1)	Реализовать нейрон для задач AND и OR. Задать случайные весовые параметры и обучить для выбранной задачи: AND или OR. Для простоты можно использовать пороговую функцию активации.
2)	Реализовать нейронную сеть из 10 нейронов, для распознавания цифр (каждый нейрон распознает только свою цифру, то есть один нейрон распознает только 0, второй 1, третий 2 и т.д.). Модифицировать первую программу для задачи AND и OR для распознавания рукописных цифр. Для повышения точности предлагается использовать функцию активации sigmoid. И скорость обучения нейронной сети. Вычислить MSE (квадратическую ошибку), в процессе обучения должна уменьшаться. Добиться точности больше 50% для всех классов цифр (случайное угадывание составляет 10%).  

Пояснение: Первая часть задания для отладки программы, на ней проще понять и отладить алгоритм. Вторая часть небольшая практическая задача, можно реализовать доработав первый алгоритм с операциями AND и OR.
Будут вопросы на понимание: что такое нейрон, как обучается нейрон, как вычислили формулы для обучения нейрона.

В помощь
Курс на Stepic: (https://stepik.org/lesson/21775/step/1?unit=5191).
Лучше начинать с главы: «2. Перцептрон и градиентный спуск». Для реализации данной лабораторной ее достаточно. (В предыдущей главе можно поизучать теорию).


## Ход работы

### Импорт библиотек

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Задание 1

In [2]:
class Perceptron:
    def __init__(self, lr: float = 0.1, epochs: int = 1000):
        self.lr = lr         # скорость обучения
        self.epochs = epochs # колличество эпох


    # В качестве функции активации возьмем сигмоиду
    def sigmoid(self, X):
        return 1 / (1 + np.exp(-X))


    # Предсказания в виде вероятности 1
    def predict_proba(self, X):
        return self.sigmoid(X @ self.w + self.b)

    # Предсказания в виде 0/1
    def predict(self, X):
        pred = self.predict_proba(X)
        return (pred >= 0.5).astype(np.uint8)

    
    def fit(self, X, y):
        # Инициализируем случайные параметры в пределах [0, 1)
        self.w = np.random.rand(X.shape[1]).astype(np.float32)
        self.b = np.float32(np.random.random())

        N = len(y)
        # так как датасет для or/and очень маленький, 
        # обучение будет проходить на всех данных сразу
        for epoch in range(self.epochs):
            pred = self.predict_proba(X) # предсказания для всего датасета
            
            # Корректируем веса
            error = pred - y
            self.w -= self.lr * X.T @ error / N
            self.b -= self.lr * error.mean()


In [3]:
features = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
labels_or = np.array([0, 1, 1, 1])
labels_and = np.array([0, 0, 0, 1])

model_or = Perceptron()
model_or.fit(features, labels_or)

model_and = Perceptron()
model_and.fit(features, labels_and)

pred_or = model_or.predict(features)
pred_and = model_and.predict(features)

print('Модель OR')
print(f'Предсказания: {pred_or}')
print(f'Истинные:     {labels_or}')
print(f'Accuracy:     {(pred_or == labels_or).mean() * 100}%')
print()
print('Модель AND')
print(f'Предсказания: {pred_and}')
print(f'Истинные:     {labels_and}')
print(f'Accuracy:     {(pred_and == labels_and).mean() * 100}%')

Модель OR
Предсказания: [0 1 1 1]
Истинные:     [0 1 1 1]
Accuracy:     100.0%

Модель AND
Предсказания: [0 0 0 1]
Истинные:     [0 0 0 1]
Accuracy:     100.0%


### Задание 2

получение данных

In [23]:
path = './data/raw'
train = pd.read_csv(f'{path}/mnist_train.csv')
test = pd.read_csv(f'{path}/mnist_test.csv')

features_train = train.drop('label', axis=1)
labels_train = train['label']

features_test = test.drop('label', axis=1)
labels_test = test['label']

features_train = features_train.to_numpy(dtype=np.float32)
labels_train = labels_train.to_numpy(dtype=np.int64)

features_test = features_test.to_numpy(dtype=np.float32)
labels_test = labels_test.to_numpy(dtype=np.uint8)

Нормализация данных

In [24]:
def normalize(X):
    return X / 255

Функция вычисления точности (accuracy)

In [60]:
def accuracy(y, y_pred):
    return np.mean((y == y_pred).astype(np.uint8))

Класс нейросети

In [61]:
class NeuralNetwork():
    def __init__(self, lr: float = 0.1, epochs: int = 10, batch_size: int = 10):
        self.lr = lr         # скорость обучения
        self.epochs = epochs # колличество эпох
        self.bs = batch_size # размер батча


    def softmax(self, X):
        # строки ниже для предотращения неопределенности inf/inf
        # в случае, если значения x будут слишком большими + или -
        max_x = np.max(X, axis=1, keepdims=True)
        exp_x = np.exp(X - max_x)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def predict_proba(self, X):
        ''' Вернет вектор вероятностей для каждой цифры '''
        # (n, 784) @ (784, 10) + (10,)
        return self.softmax(X @ self.w.T + self.b)


    def predict(self, X):
        ''' Вернет предсказание в виде наиболее вероятной цифры '''
        return np.argmax(self.predict_proba(X), axis=1)

    
    def cross_entropy(self, P, Y):
        eps = 1e-12 # чтобы log(0) не ушел к inf
        return -np.sum(Y * np.log(P + eps), axis=1)


    def train(self, X, y):
        n, d = X.shape
        # w - матрица, где строки это векторы весов каждого нейрона
        self.w = (np.random.randn(10, d) * 0.1).astype(np.float32) # (10, d)
        # b - вектор из смещений для каждого нейрона
        self.b = np.zeros(10, dtype=np.float32)                    # (10,)

        # получаем вектор с 1 в индексе цифры
        Y = np.eye(10, dtype=np.float32)[y]

        for epoch in range(self.epochs):
            # перемешиваем индексы
            idx = np.random.permutation(n)

            epoch_loss = 0.0
            for i in range(0, n, self.bs):
                # получаем батч
                batch_indexes = idx[i:i + self.bs]
                X_batch = X[batch_indexes]      # (B, 784)
                Y_batch = Y[batch_indexes]      # (B, 10)
                B = X_batch.shape[0]

                # Прямой проход
                Z = X_batch @ self.w.T + self.b # (B, 10)
                P = self.softmax(Z)             # (B, 10)

                # Обратный проход
                dZ = (P - Y_batch) / B          # (B, 10)
                dw = dZ.T @ X_batch             # (10, d)
                db = dZ.sum(axis=0)             # (10,)

                self.w -= self.lr * dw
                self.b -= self.lr * db

                epoch_loss += self.cross_entropy(P, Y_batch).sum()
            epoch_loss /= n
            acc = (self.predict(X) == y).mean()
            print(f'Эпоха {epoch + 1}: loss={epoch_loss:.4f}, acc={acc:.4f}')


Обучение сети

In [62]:
nn = NeuralNetwork(batch_size=1000, lr=0.1, epochs=50)
nn.train(normalize(features_train), labels_train)

Эпоха 1: loss=1.2336, acc=0.8041
Эпоха 2: loss=0.6703, acc=0.8457
Эпоха 3: loss=0.5546, acc=0.8623
Эпоха 4: loss=0.4987, acc=0.8720
Эпоха 5: loss=0.4648, acc=0.8782
Эпоха 6: loss=0.4412, acc=0.8823
Эпоха 7: loss=0.4239, acc=0.8863
Эпоха 8: loss=0.4103, acc=0.8894
Эпоха 9: loss=0.3993, acc=0.8917
Эпоха 10: loss=0.3902, acc=0.8937
Эпоха 11: loss=0.3825, acc=0.8952
Эпоха 12: loss=0.3758, acc=0.8971
Эпоха 13: loss=0.3700, acc=0.8988
Эпоха 14: loss=0.3649, acc=0.8999
Эпоха 15: loss=0.3602, acc=0.9011
Эпоха 16: loss=0.3561, acc=0.9017
Эпоха 17: loss=0.3523, acc=0.9027
Эпоха 18: loss=0.3489, acc=0.9033
Эпоха 19: loss=0.3457, acc=0.9043
Эпоха 20: loss=0.3429, acc=0.9045
Эпоха 21: loss=0.3402, acc=0.9053
Эпоха 22: loss=0.3377, acc=0.9057
Эпоха 23: loss=0.3353, acc=0.9063
Эпоха 24: loss=0.3331, acc=0.9071
Эпоха 25: loss=0.3311, acc=0.9079
Эпоха 26: loss=0.3292, acc=0.9080
Эпоха 27: loss=0.3274, acc=0.9089
Эпоха 28: loss=0.3257, acc=0.9092
Эпоха 29: loss=0.3240, acc=0.9096
Эпоха 30: loss=0.3224, 

Проверка на test

In [63]:
pred = nn.predict(features_test)
print(f'Accuracy: {accuracy(labels_test, pred)}')

Accuracy: 0.9011
